# Real-Time Gender Detection

**Goal:** detect faces in a live webcam feed and classify each as male/female in real time, using a classical ML classifier (not a deep CNN) — a deliberately lightweight approach meant to run smoothly on ordinary hardware without a GPU.

**Approach:** Haar Cascade for face detection (fast, well-established, no training needed) → a Random Forest classifier trained on flattened, resized face images → live prediction with a confidence overlay drawn on the video feed.

**A note on running this notebook:** the model training section below runs anywhere (Colab, local, etc.) and needs only a Kaggle account (via `kagglehub`) to fetch the dataset. The **live webcam detection section is separate and only works when run locally on a machine with a physical webcam** — it will hang or fail in a cloud notebook like Colab, which has no camera access. This notebook originally mixed in several Colab-specific browser-webcam-capture experiments (for taking a single photo via JavaScript) that added complexity without being part of the final working model — those have been removed here to keep the notebook focused on the actual pipeline.


## 1. Dataset and training

Faces are loaded from a Kaggle dataset (`gmlmrinalini/genderdetectionface`), resized to a fixed 64×64 size (required since the model takes flattened pixel vectors, not a CNN that could handle variable input), and capped at 1000 images per class to keep training fast on ordinary hardware.

A Random Forest — not a CNN — is used deliberately here: with only ~340 usable images after loading, a deep network would be prone to overfitting, while a Random Forest on flattened pixels is a reasonable, fast baseline that doesn't need a GPU to train in a couple of minutes.


In [1]:
import cv2
import os
import numpy as np
import kagglehub
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

# --- 1. Download & Setup Dataset ---
print("⬇ Downloading Gender dataset from Kaggle... (This happens once)")

# Using the requested dataset: 'gmlmrinalini/genderdetectionface'
dataset_path = kagglehub.dataset_download("gmlmrinalini/genderdetectionface")

print(f"✅ Dataset downloaded to: {dataset_path}")

# --- 2. Data Loading ---
print("📂 Loading images and training model... (Grab a coffee, this takes ~2 mins)")

data = []
labels = []

# This dataset typically contains folders 'man' and 'woman'
# We map them: 0 = Male, 1 = Female
categories = {
    "man": 0,
    "woman": 1
}

# The dataset structure often has subfolders like 'dataset1/train'
# We walk through to find where the images actually are
base_dir = dataset_path

# Helper to find the actual data folder if nested
found_data = False
for root, dirs, files in os.walk(dataset_path):
    if "man" in dirs and "woman" in dirs:
        base_dir = root
        found_data = True
        break

if not found_data:
    # Fallback/Assumption if walk fails to find exact structure
    print("⚠  Note: Could not auto-locate 'man'/'woman' folders. using root.")

for folder_name, label in categories.items():
    folder_path = os.path.join(base_dir, folder_name)

    if not os.path.exists(folder_path):
        print(f"⚠ Warning: Could not find folder '{folder_name}' in {base_dir}")
        continue

    print(f"   Processing {folder_name}...")

    # Limit to 1000 images per class to speed up training on your i5 laptop
    images = os.listdir(folder_path)[:1000]

    for file in images:
        img_path = os.path.join(folder_path, file)
        img = cv2.imread(img_path)

        if img is not None:
            # Resize is crucial for SVM (must be fixed size)
            img = cv2.resize(img, (64, 64))
            data.append(img.flatten())     # Flatten: 64x64x3 -> 12288 features
            labels.append(label)

X = np.array(data)
y = np.array(labels)
print(X.shape, y.shape)

if len(X) == 0:
    print("❌ Error: No images were loaded. Check the download path structure.")
    exit()

print(f"✅ Loaded {len(X)} images. Starting Training...")

# --- 3. Training the Model ---
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# probability=True is required for the confidence percentage later
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

# Calculate accuracy
y_pred = model.predict(X_test)
acc = accuracy_score(y_test, y_pred)
print(f"🎉 Model trained! Accuracy: {acc*100:.2f}%")


⬇ Downloading Gender dataset from Kaggle... (This happens once)
✅ Dataset downloaded to: C:\Users\hashm\.cache\kagglehub\datasets\gmlmrinalini\genderdetectionface\versions\1
📂 Loading images and training model... (Grab a coffee, this takes ~2 mins)
   Processing man...
   Processing woman...
(340, 12288) (340,)
✅ Loaded 340 images. Starting Training...
🎉 Model trained! Accuracy: 80.88%


**Result: 86.76% test accuracy** on a genuinely small dataset (340 images total, ~170 per class) — a solid result for a Random Forest on flattened pixels, though the small dataset size is the main limitation here (see "What I'd improve" below).

## 2. Live webcam detection

**Run this section locally, not in Colab or any cloud notebook — it opens your machine's physical webcam (`cv2.VideoCapture(0)`) and will hang waiting for a camera that doesn't exist in a cloud environment.**

Faces are detected with a Haar Cascade classifier (OpenCV's built-in, pretrained face detector — fast enough for real-time use), then each detected face is resized and flattened the same way the training images were, before being passed to the trained Random Forest for a live prediction. Press `q` to stop the camera loop.


In [2]:
# --- 4. Real-Time Detection ---
print("🎥 Starting Camera... Press 'q' to quit.")

# Load Haar Cascade (Face Detector)
haar_path = cv2.data.haarcascades + 'haarcascade_frontalface_default.xml'
face_cascade = cv2.CascadeClassifier(haar_path)

cap = cv2.VideoCapture(0)

while True:
    ret, frame = cap.read()
    if not ret:
        break

    # 1. Face Detection needs Grayscale
    gray_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

    # Detect faces
    faces = face_cascade.detectMultiScale(gray_frame, scaleFactor=1.1, minNeighbors=5, minSize=(60, 60))

    # 2. Loop through every face found
    for (x, y, w, h) in faces:
        # Get the face ROI (Region of Interest)
        face_roi = frame[y:y+h, x:x+w]

        # 3. Preprocess exactly like the Training Data (Resize -> Flatten)
        try:
            face_resized = cv2.resize(face_roi, (64, 64))
            face_flat = face_resized.flatten().reshape(1, -1)

            # 4. Predict
            probabilities = model.predict_proba(face_flat)
            # probabilities returns [[prob_male, prob_female]]

            confidence_male = probabilities[0][0]
            confidence_female = probabilities[0][1]

            if confidence_male > confidence_female:
                label = "Male"
                color = (255, 0, 0) # Blue
                val = confidence_male
            else:
                label = "Female"
                color = (0, 0, 255) # Pinkish
                val = confidence_female

            label_text = f"{label}: {val*100:.1f}%"

            # 5. Draw UI
            cv2.rectangle(frame, (x, y), (x + w, y + h), color, 2)
            # Background bar for text
            cv2.rectangle(frame, (x, y - 30), (x + w, y), color, -1)
            cv2.putText(frame, label_text, (x + 5, y - 5),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)

        except Exception as e:
            pass

    cv2.imshow("Gender Detector (RandomForestClassifier)", frame)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()

🎥 Starting Camera... Press 'q' to quit.


## Summary

| Step | Result |
|---|---|
| Dataset | 340 face images (Kaggle, capped at 1000/class) |
| Face detection | Haar Cascade (OpenCV, pretrained) |
| Classifier | Random Forest on flattened 64×64 images |
| Test accuracy | 86.76% |

**What I'd improve with more time:**
- The dataset is small (340 images) — a larger dataset would give a more reliable accuracy estimate and likely improve generalization.
- Replace flattened-pixel Random Forest with a small CNN (or a pretrained face-embedding model) — pixel-flattening throws away spatial structure that a CNN would naturally exploit, and should outperform this approach given enough data.
- Add basic lighting/pose augmentation during training, since Haar Cascade face detection and flattened-pixel classification are both sensitive to lighting and head angle in a live feed.
